# Laboratorio UNIR — Evaluación de la segmentación

Comparación de Otsu y K-means sobre tres escenas sintéticas con máscaras ground truth.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from segmentation_lab import otsu_threshold, kmeans_segment, binary_metrics, confusion_matrix
if not list((ROOT / 'data/images').glob('*.png')):
    exec((ROOT / 'scripts/generate_dataset.py').read_text(encoding='utf-8'))


In [ ]:
rows = []
for image_path in sorted((ROOT / 'data/images').glob('*.png')):
    name = image_path.stem
    image = np.asarray(Image.open(image_path).convert('L'))
    truth = np.asarray(Image.open(ROOT / 'data/masks' / f'{name}_mask.png').convert('L')) > 0
    threshold, otsu_mask = otsu_threshold(image)
    centroids, kmeans_mask = kmeans_segment(image, seed=7)
    rows.append({'image': name, 'method': 'Otsu', 'parameter': f'threshold={threshold}', **binary_metrics(truth, otsu_mask)})
    rows.append({'image': name, 'method': 'K-means', 'parameter': f'centroids={np.round(centroids, 2)}', **binary_metrics(truth, kmeans_mask)})
rows


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for row, image_path in enumerate(sorted((ROOT / 'data/images').glob('*.png'))):
    name = image_path.stem
    image = np.asarray(Image.open(image_path).convert('L'))
    truth = np.asarray(Image.open(ROOT / 'data/masks' / f'{name}_mask.png').convert('L')) > 0
    _, otsu_mask = otsu_threshold(image); _, kmeans_mask = kmeans_segment(image, seed=7)
    for axis, data, title in zip(axes[row], [image, truth, otsu_mask, kmeans_mask], ['Imagen', 'Ground truth', 'Otsu', 'K-means']):
        axis.imshow(data, cmap='gray'); axis.set_title(f'{name}: {title}'); axis.axis('off')
plt.tight_layout(); plt.show()


## Evidencia

La tabla `rows` contiene IoU, Dice, precisión y recall. La matriz de confusión se expresa como `[[TN, FP], [FN, TP]]` y se calcula explícitamente con `confusion_matrix`.


In [ ]:
for image_path in sorted((ROOT / 'data/images').glob('*.png')):
    name = image_path.stem; image = np.asarray(Image.open(image_path).convert('L'))
    truth = np.asarray(Image.open(ROOT / 'data/masks' / f'{name}_mask.png').convert('L')) > 0
    threshold, prediction = otsu_threshold(image)
    print(name, 'Otsu', 'threshold=', threshold, 'confusion=', confusion_matrix(truth, prediction).tolist())
